# ⚡ HELIOS AI COGNITIVE RUNTIME — H100 GPU 2B-Token Accelerator
### **High-Throughput Pretraining (120k–180k tok/s), Zero-Loss Checkpointing & HF/Ollama Sync**

**Architect & Creator**: Arron Kian Parejas — Creator of Helios & Domodomo · Top 1 on AppBuilders · Former GDG Chapter Lead (Philippines)  

---

### **Key H100 Hardware Accelerations**
1. **NVIDIA H100 SXM5 80GB**: Native **bfloat16 precision** + `torch.compile(mode="reduce-overhead")` + PyTorch SDPA kernel fusion.
2. **Continuous Google Drive Auto-Save**: Direct streaming to `/content/drive/MyDrive/Helios/`.
3. **Compute-Unit Preemption Shield**: Traps `SIGTERM`/`SIGINT`. Immediate checkpoint flush on disconnect.
4. **Live Hugging Face Hub Sync**: Automatic model upload to `Darknecrocities/Helios`.
5. **Auto-Eviction**: Automatic cleanup of `/tmp` and local cache to prevent Colab disk crashes.

## 1. Verify NVIDIA H100 GPU Hardware & FlashAttention / BF16

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Model: {gpu_name}")
    print(f"Total VRAM: {vram:.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")
    print(f"TF32 Allowed: {torch.backends.cuda.matmul.allow_tf32}")

## 2. Mount Google Drive for Zero-Loss Checkpoint Persistence

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Helios')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'checkpoints').mkdir(exist_ok=True)
(DRIVE_ROOT / 'manifests').mkdir(exist_ok=True)
(DRIVE_ROOT / 'datasets').mkdir(exist_ok=True)
print(f"Google Drive Helios Root Ready: {DRIVE_ROOT}")

## 3. Clone Repository & Set Hugging Face Credentials

In [ ]:
import os
%cd /content
!git clone -b pretrained https://github.com/darknecrocities/Helios.git helios_repo || (cd helios_repo && git pull origin pretrained)
%cd /content/helios_repo
!pip install -q -e .
!pip install -q safetensors rich tokenizers datasets huggingface_hub

# Optional: Set your Hugging Face Token for continuous hub sync
# os.environ["HF_TOKEN"] = "hf_..."
print("Helios H100 environment initialized.")

## 4. Checkpoint Health & Resumed Ledger Cursor

In [ ]:
import json

ledger_file = DRIVE_ROOT / 'manifests' / 'curriculum_ledger.json'
if ledger_file.exists():
    with open(ledger_file, 'r') as f:
        ledger = json.load(f)
    print("Existing curriculum ledger detected:")
    print(f"  - Cursor Position: {ledger.get('category_cursors', {}).get('all', 0)}")
    print(f"  - Active Epoch: {ledger.get('category_epochs', {}).get('all', 1)}")
    print(f"  - Deduplicated Hashes: {len(ledger.get('seen_hashes', []))}")
else:
    print("Starting fresh ledger at cursor 0.")

for tier in ['nano', 'alpha', 'beta', 'core', 'pro', 'flagship']:
    st = DRIVE_ROOT / 'checkpoints' / f"{tier}_continuous" / 'latest' / 'model.safetensors'
    status = f"EXISTS ({st.stat().st_size / 1e6:.1f} MB)" if st.exists() else "INITIALIZING"
    print(f"Tier {tier.upper():<10}: {status}")

## 5. Launch H100 2B-Token Pretraining Runner
Executes with torch.compile, bfloat16, FlashAttention, and automatic checkpointing to Drive.

In [ ]:
!python scripts/colab_runner.py \
    --drive-root "/content/drive/MyDrive/Helios" \
    --target-tokens 2000000000 \
    --batch-size 64 \
    --block-size 1024

## 6. Interactive Verification & Deductive Reasoning Test

In [ ]:
!python -c "
from helios.cli.fallback import get_grounded_fallback
queries = [
    'who is napoleon?',
    'who is napoleon bonaparte?',
    'what is europe?',
    'who is donald trump?',
    'what is photosynthesis?'
]
for q in queries:
    ans = get_grounded_fallback(q, tier='nano-20m')
    print(f'\n[Query]: {q}')
    print(f'[Helios Response]: {ans}')
"